In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')
df_silver.show()

In [0]:
df_silver.printSchema()

In [0]:
df_gold = df_silver.select('customer_id', 'customer_name', 'city', "customer",  "market", "platform", "channel")

df_gold.show()

In [0]:
df_gold.write.format("delta").option('delta.enableChangeDataFeed', 'true').mode('overwrite').saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [0]:
delta_table = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.dim_{data_source}')

child_customer_table = spark.table(f'{catalog}.{gold_schema}.sb_dim_{data_source}').select(
    F.col('customer_id').alias('customer_code'),
    'customer',
    'market',
    'platform',
    'channel'

)

In [0]:
delta_table.alias('target').merge(
    source = child_customer_table.alias('source'),
    condition = 'target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#sanity check
display(spark.sql(f'SELECT * FROM {catalog}.{gold_schema}.dim_{data_source}'))